# codon-infer · Exportación canónica de pesos ESM-2 + referencias HF

Genera para cada escala:
- `esm2_<escala>_weights.bin` en **layout agrupado canónico** (FP32 little-endian, sin `inv_freq` ni `contact_head`).
- `ref_<escala>.npz` con los hidden states de cada capa y los logits de HuggingFace sobre `test_peptidase` (la verdad de referencia para los tests de Rust).
- `MANIFEST.json` con el tamaño, el tamaño esperado y el sha256 de cada archivo.

**Entorno:** `Entorno de ejecución → Cambiar tipo` → **High-RAM** (imprescindible para 3B). Los archivos se escriben directamente en Google Drive (`MyDrive/codon_infer_weights_v2/`).

**Orden recomendado:** ejecuta primero solo 8M y valídalo en el repo. Después, el resto.

In [ ]:
# Celda 1 · Dependencias y Drive
!pip -q install "transformers>=4.40" safetensors numpy
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/codon_infer_weights_v2'
os.makedirs(OUT_DIR, exist_ok=True)
!free -g
print('Salida:', OUT_DIR)

In [ ]:
# Celda 2 · Definiciones (no descarga nada)
import torch, numpy as np, hashlib, json, gc, time
from transformers import AutoModelForMaskedLM, AutoTokenizer

torch.set_grad_enabled(False)

MODELS = {
    '8M':   ('facebook/esm2_t6_8M_UR50D',    6,  320,  1280),
    '35M':  ('facebook/esm2_t12_35M_UR50D',  12, 480,  1920),
    '150M': ('facebook/esm2_t30_150M_UR50D', 30, 640,  2560),
    '650M': ('facebook/esm2_t33_650M_UR50D', 33, 1280, 5120),
    '3B':   ('facebook/esm2_t36_3B_UR50D',   36, 2560, 10240),
}
REF_SEQ = 'AQVGDSGGPMLCNGQLQGVTSWGYVCAQPEYPGVYTEVS'  # test_peptidase.fasta (39 aa)
V = 33

PER_LAYER = [  # orden agrupado: por tipo de tensor y, dentro, por capa
    'attention.LayerNorm.weight', 'attention.LayerNorm.bias',
    'attention.self.query.weight', 'attention.self.query.bias',
    'attention.self.key.weight',   'attention.self.key.bias',
    'attention.self.value.weight', 'attention.self.value.bias',
    'attention.output.dense.weight', 'attention.output.dense.bias',
    'LayerNorm.weight', 'LayerNorm.bias',
    'intermediate.dense.weight', 'intermediate.dense.bias',
    'output.dense.weight', 'output.dense.bias',
]

def expected_size(L, d, h):
    core = V*d + L*(2*d + 4*(d*d + d) + 2*d + d*h + h + h*d + d) + 2*d
    head = 2*d + d*d + d + V*d + V
    return (core + head) * 4

def sha256(path):
    hs = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            hs.update(b)
    return hs.hexdigest()

def write_t(f, t):
    t.detach().cpu().contiguous().numpy().astype('<f4', copy=False).tofile(f)

def process(name, make_ref=True):
    hf, L, d, h = MODELS[name]
    tag = name.lower()
    bin_path = f'{OUT_DIR}/esm2_{tag}_weights.bin'
    ref_path = f'{OUT_DIR}/ref_{tag}.npz'
    t0 = time.time()
    print(f'\n=== {name}: cargando {hf} ===')
    model = AutoModelForMaskedLM.from_pretrained(hf, torch_dtype=torch.float32, low_cpu_mem_usage=True).eval()
    cfg = model.config
    assert (cfg.num_hidden_layers, cfg.hidden_size, cfg.intermediate_size, cfg.vocab_size) == (L, d, h, V), cfg
    sd = model.state_dict()
    emb = sd['esm.embeddings.word_embeddings.weight']
    dec = sd.get('lm_head.decoder.weight', emb)

    # --- binario agrupado canónico ---
    with open(bin_path, 'wb') as f:
        write_t(f, emb)
        for k in PER_LAYER:
            for l in range(L):
                write_t(f, sd[f'esm.encoder.layer.{l}.{k}'])
        write_t(f, sd['esm.encoder.emb_layer_norm_after.weight'])
        write_t(f, sd['esm.encoder.emb_layer_norm_after.bias'])
        for k in ['lm_head.layer_norm.weight', 'lm_head.layer_norm.bias',
                  'lm_head.dense.weight', 'lm_head.dense.bias']:
            write_t(f, sd[k])
        write_t(f, dec)
        write_t(f, sd['lm_head.bias'])
    real, exp = os.path.getsize(bin_path), expected_size(L, d, h)
    assert real == exp, f'{name}: tamaño {real} != esperado {exp}'
    digest = sha256(bin_path)
    print(f'{name}: {real:,} B (OK) sha256={digest}')

    # --- referencia HF ---
    ref_info = None
    if make_ref:
        tok = AutoTokenizer.from_pretrained(hf)
        enc = tok(REF_SEQ, return_tensors='pt')
        out = model(**enc, output_hidden_states=True)
        logits = out.logits[0].float().numpy()
        hidden = np.stack([hs[0].float().numpy() for hs in out.hidden_states])  # [L+1, T, d]
        ids = enc['input_ids'][0].numpy()
        pred = logits.argmax(-1)
        top1 = float((pred[1:-1] == ids[1:-1]).mean())
        np.savez_compressed(ref_path, input_ids=ids, logits=logits, hidden_states=hidden,
                            sequence=np.array(REF_SEQ))
        print(f'{name}: ref guardada, Top-1 sin máscara = {top1:.2%}')
        ref_info = {'file': os.path.basename(ref_path), 'sha256': sha256(ref_path), 'top1_unmasked': top1,
                    'note': 'hidden_states[0]=embedding, hidden_states[-1]=tras emb_layer_norm_after'}

    entry = {'model': hf, 'layout': 'grouped_canonical_v1', 'file': os.path.basename(bin_path),
             'size': real, 'expected_size': exp, 'sha256': digest,
             'n_layers': L, 'dim': d, 'hidden_dim': h, 'vocab_size': V, 'ref': ref_info,
             'transformers': __import__('transformers').__version__, 'seconds': round(time.time() - t0, 1)}
    man_path = f'{OUT_DIR}/MANIFEST.json'
    manifest = json.load(open(man_path)) if os.path.exists(man_path) else {}
    manifest[name] = entry
    json.dump(manifest, open(man_path, 'w'), indent=2)
    del model, sd, emb, dec
    gc.collect()
    return entry

print('Listo. Tamaños esperados:')
for n, (_, L, d, h) in MODELS.items():
    print(f'  {n:>5}: {expected_size(L, d, h):,} B')

In [ ]:
# Celda 3 · Control: 8M (valídalo en el repo ANTES de seguir)
process('8M')

In [ ]:
# Celda 4 · 35M, 150M, 650M
for n in ['35M', '150M', '650M']:
    process(n)

In [ ]:
# Celda 5 · 3B (requiere High-RAM; descarga ~11 GB, tarda bastante)
!free -g
process('3B')

In [ ]:
# Celda 6 · Resumen final
import json
print(json.dumps(json.load(open(f'{OUT_DIR}/MANIFEST.json')), indent=2))
!ls -l {OUT_DIR}